# BTC Long-Term Price Forecast

Calibrated statistical forecasts for BTC price action at **1-year, 2-year, and 4-year** horizons (~next halving cycle).

**Features used:**
- OHLCV daily candles (from local DuckDB via `ccquant`)
- Market-cap weightings / BTC dominance (CoinGecko live + volume-weighted proxy from DB)
- Halving cycle progress (supply-schedule structural feature)
- Network hashrate (blockchain.info, live)
- FRED monetary indicators: M2 money stock, Fed balance sheet, 10Y Treasury yield (requires `FRED_API_KEY`)

**Models:**
1. **Cointegrating OLS** — log(BTC) ~ time + log(M2) + log(hashrate) + halving-cycle progress, with HAC (Newey-West) standard errors
2. **ARIMA** baseline on log(price)

**Calibration:** rolling-origin backtest + conformal rescaling so that stated 50%/95% intervals achieve their nominal coverage empirically.

> **Run order is top-to-bottom.** Set `FRED_API_KEY` env var before launching to enable the monetary-features block; without it the notebook degrades gracefully (model runs on OHLCV + halving + hashrate only).

In [1]:
from __future__ import annotations

import os
import warnings
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import httpx
import numpy as np
import plotly.graph_objects as go
import polars as pl
import statsmodels.api as sm
from dotenv import load_dotenv
from statsmodels.tsa.arima.model import ARIMA

from ccquant.forecasting import load_daily_panel

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Config ----------------------------------------------------------------
# Resolve project root (works whether cwd is repo root or notebooks/)
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

# Auto-load .env from project root (FRED_API_KEY, CCQUANT_DB, etc.)
load_dotenv(_root / ".env")

DB_PATH = os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb"))
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
HORIZONS = [365, 730, 1461]          # 1y, 2y, ~4y (next halving cycle)
BOOTSTRAP_DRAWS = 500
CALIB_BOOTSTRAP_DRAWS = 200
CALIB_STEP_DAYS = 90
RNG_SEED = 42

# Bitcoin halving dates (block subsidy cuts)
HALVING_DATES = [
    date(2012, 11, 28),
    date(2016, 7, 9),
    date(2020, 5, 11),
    date(2024, 4, 20),
]
NEXT_HALVING_EST = date(2028, 4, 1)   # approximate

print(f"DB: {DB_PATH}")
print(f"FRED_API_KEY set: {bool(FRED_API_KEY)}")
print(f"Horizons (days): {HORIZONS}")

DB: /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
FRED_API_KEY set: True
Horizons (days): [365, 730, 1461]


## 1. Load BTC OHLCV from DuckDB

Zero-copy Arrow path (`to_arrow_table` → `pl.from_arrow`) — ~4x faster than row-wise `fetchall` on the full table.

In [2]:
panel = load_daily_panel(DB_PATH)
print(f"Full panel: {panel.shape}  symbols: {panel['symbol'].n_unique()}")

btc = (
    panel.filter(pl.col("symbol") == "BTC")
    .sort("date")
    .with_columns(np.log(pl.col("close")).alias("log_close"))
)
print(f"BTC daily: {btc.height} rows  {btc['date'].min()} → {btc['date'].max()}")
btc.tail(5)

Full panel: (70344, 8)  symbols: 47
BTC daily: 4001 rows  2015-07-20 → 2026-07-02


symbol,date,open,high,low,close,volume,source,log_close
str,date,f64,f64,f64,f64,f64,str,f64
"""BTC""",2026-06-28,59934.84,60455.0,58810.11,59474.01,4245.962419,"""coinbase""",10.993295
"""BTC""",2026-06-29,59473.99,60702.63,58800.0,60162.73,10607.132609,"""coinbase""",11.004808
"""BTC""",2026-06-30,60162.73,60191.99,58056.0,58523.93,9465.932327,"""coinbase""",10.977191
"""BTC""",2026-07-01,58523.93,61287.33,57717.55,59961.45,10684.762987,"""coinbase""",11.001457
"""BTC""",2026-07-02,59961.46,62147.89,59520.02,61646.26,7908.669911,"""coinbase""",11.029168


## 2. Market-Cap Weightings & BTC Dominance

Two views:
- **Live** market-cap weights from CoinGecko (current snapshot of the top universe).
- **Historical volume-weighted BTC dominance** proxy computed from the local OHLCV cross-section (volume × close as a market-cap stand-in, since we don't store historical market caps).

In [3]:
# --- Live market-cap weights from CoinGecko -------------------------------
def fetch_market_caps(size: int = 100) -> list[dict]:
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(
            "https://api.coingecko.com/api/v3/coins/markets",
            params={
                "vs_currency": "usd",
                "order": "market_cap_desc",
                "per_page": min(100, size),
                "page": 1,
                "sparkline": "false",
            },
        )
        resp.raise_for_status()
        return resp.json()

mkt_data = fetch_market_caps(size=100)
mkt_caps = pl.DataFrame({
    "symbol": [d["symbol"].upper() for d in mkt_data],
    "market_cap": [float(d.get("market_cap") or 0) for d in mkt_data],
    "rank": [d["market_cap_rank"] or 0 for d in mkt_data],
}).sort("market_cap", descending=True)

total_mc = mkt_caps["market_cap"].sum()
mkt_caps = mkt_caps.with_columns(
    (pl.col("market_cap") / total_mc * 100).round(2).alias("weight_pct")
)
btc_weight = float(mkt_caps.filter(pl.col("symbol") == "BTC")["weight_pct"][0])
print(f"BTC market-cap weight in top-100: {btc_weight:.1f}%")
print("Top 10 by market cap:")
mkt_caps.head(10)

BTC market-cap weight in top-100: 57.7%
Top 10 by market cap:


symbol,market_cap,rank,weight_pct
str,f64,i64,f64
"""BTC""",1.2784e12,1,57.69
"""ETH""",2.1316e11,2,9.62
"""USDT""",1.8416e11,3,8.31
"""BNB""",7.7510e10,4,3.5
"""USDC""",7.3327e10,5,3.31
"""XRP""",6.8664e10,6,3.1
"""SOL""",4.5814e10,7,2.07
"""TRX""",3.1455e10,8,1.42
"""FIGR_HELOC""",1.9790e10,9,0.89


In [4]:
# --- Historical volume-weighted BTC dominance proxy (from DB) ------------
# Proxy: daily close * volume as a market-cap stand-in across the universe.
universe = panel.filter(pl.col("symbol").is_in(mkt_caps["symbol"].to_list()))
daily_mc_proxy = (
    universe.with_columns((pl.col("close") * pl.col("volume")).alias("mc_proxy"))
    .group_by("date")
    .agg(
        pl.col("mc_proxy").filter(pl.col("symbol") == "BTC").sum().alias("btc_mc"),
        pl.col("mc_proxy").sum().alias("total_mc"),
    )
    .sort("date")
    .with_columns(
        (pl.col("btc_mc") / pl.col("total_mc") * 100).alias("btc_dominance_pct")
    )
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_mc_proxy["date"], y=daily_mc_proxy["btc_dominance_pct"],
    mode="lines", name="BTC dominance (vol×price proxy)", line=dict(color="#f7931a"),
))
fig.update_layout(
    title="BTC Dominance Proxy (volume-weighted, from local OHLCV)",
    yaxis_title="%", height=400, template="plotly_dark",
)
fig.show()

print(f"Current proxy dominance: {daily_mc_proxy['btc_dominance_pct'][-1]:.1f}%")
print(f"Mean: {daily_mc_proxy['btc_dominance_pct'].mean():.1f}%  "
      f"Min: {daily_mc_proxy['btc_dominance_pct'].min():.1f}%  "
      f"Max: {daily_mc_proxy['btc_dominance_pct'].max():.1f}%")

Current proxy dominance: 46.9%
Mean: 51.2%  Min: 9.8%  Max: 100.0%


## 3. Halving Cycle Feature

Each halving cuts the block subsidy in half, creating a ~4-year supply-shock cycle. We encode **cycle progress** = fraction of the current halving epoch elapsed (0 at halving → 1 at next halving).

In [5]:
def halving_cycle_progress(d: date) -> float:
    """Return progress 0..1 within the current halving epoch."""
    prev = max(h for h in HALVING_DATES if h <= d) if any(h <= d for h in HALVING_DATES) else HALVING_DATES[0]
    nxt = NEXT_HALVING_EST
    for i, h in enumerate(HALVING_DATES):
        if h == prev and i + 1 < len(HALVING_DATES):
            nxt = HALVING_DATES[i + 1]
    total = (nxt - prev).days
    elapsed = (d - prev).days
    return min(max(elapsed / total, 0.0), 1.0) if total > 0 else 0.0

btc = btc.with_columns(
    pl.col("date").map_elements(halving_cycle_progress, return_dtype=pl.Float64).alias("halving_progress")
)

# Mark halvings on the price chart
fig = go.Figure()
fig.add_trace(go.Scatter(x=btc["date"], y=btc["close"], mode="lines",
                         name="BTC close", line=dict(color="#f7931a", width=1.5)))
for h in HALVING_DATES:
    fig.add_vline(x=h.isoformat(), line_dash="dash", line_color="cyan",
                  annotation_text=f"halving {h.year}", annotation_textangle=-90)
fig.update_layout(yaxis_type="log", title="BTC price (log) with halvings",
                  height=450, template="plotly_dark")
fig.show()

## 4. Network Hashrate (blockchain.info, live)

Hashrate proxies network security / miner capital expenditure / commitment. Fetched live from the blockchain.info charts API (no key required).

In [6]:
def fetch_hashrate() -> pl.DataFrame:
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(
            "https://api.blockchain.info/charts/hash-rate",
            params={"timespan": "all", "format": "json"},
        )
        resp.raise_for_status()
        data = resp.json()
    vals = data["values"]
    return pl.DataFrame({
        "date": [datetime.fromtimestamp(v["x"], tz=UTC).date() for v in vals],
        "hashrate": [float(v["y"]) for v in vals],
    }).sort("date")

hashrate = fetch_hashrate()
print(f"Hashrate: {hashrate.height} rows  {hashrate['date'].min()} → {hashrate['date'].max()}")
hashrate = hashrate.with_columns(np.log(pl.col("hashrate") + 1e-12).alias("log_hashrate"))
hashrate.tail(3)

Hashrate: 1599 rows  2009-01-03 → 2026-07-08


date,hashrate,log_hashrate
date,f64,f64
2026-06-30,9.9155e8,20.714781
2026-07-04,1.0248e9,20.747788
2026-07-08,8.9173e8,20.608675


## 5. FRED Monetary Indicators (requires `FRED_API_KEY`)

Fetched directly via the FRED REST API (no extra dependencies):
- **M2SL** — M2 money stock (monthly) → log-transformed
- **WALCL** — Fed total assets / balance sheet (weekly) → log-transformed
- **DGS10** — 10-Year Treasury yield (daily)

If `FRED_API_KEY` is not set, this block degrades gracefully: a synthetic M2 trend is generated from public nominal-GDP growth rates so the notebook still runs end-to-end. **Set the key for real forecasts.**

In [7]:
FRED_SERIES = {
    "M2SL": {"freq": "monthly", "transform": "log"},
    "WALCL": {"freq": "weekly", "transform": "log"},
    "DGS10": {"freq": "daily", "transform": "level"},
}

def fetch_fred_series(series_id: str, api_key: str, start: str) -> pl.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id, "api_key": api_key, "file_type": "json",
        "observation_start": start,
    }
    with httpx.Client(timeout=30.0) as client:
        resp = client.get(url, params=params)
        resp.raise_for_status()
        obs = resp.json()["observations"]
    rows = []
    for o in obs:
        v = o["value"]
        if v != "." and v:
            rows.append({"date": date.fromisoformat(o["date"]), "value": float(v)})
    return pl.DataFrame(rows).sort("date").rename({"value": series_id})

fred_frames: dict[str, pl.DataFrame | None] = {}

if FRED_API_KEY:
    start_date = (btc["date"].min() - timedelta(days=30)).isoformat()
    print("Fetching FRED series...")
    for sid in FRED_SERIES:
        try:
            df = fetch_fred_series(sid, FRED_API_KEY, start_date)
            fred_frames[sid] = df
            print(f"  {sid}: {df.height} obs")
        except Exception as exc:
            print(f"  {sid}: FAILED ({exc})")
            fred_frames[sid] = None
else:
    print("FRED_API_KEY not set — generating synthetic M2 trend (set the key for real data).")
    fred_frames = {sid: None for sid in FRED_SERIES}

Fetching FRED series...
  M2SL: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=M2SL&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3A%2F%2Ffred.stlouisfed.org%2Fdocs%2Fapi%2Fapi_key.html&file_type=json&observation_start=2015-06-20'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400)
  WALCL: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=WALCL&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3A%2F%2Ffred.stlouisfed.org%2Fdocs%2Fapi%2Fapi_key.html&file_type=json&observation_start=2015-06-20'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400)
  DGS10: FAILED (Client error '400 Bad Request' for url 'https://api.stlouisfed.org/fred/series/observations?series_id=DGS10&api_key=e4ad1d8de793173063dcdcb6ee75da5a++++++++++%23+https%3A%2F%2Ffred.stlouisfed.org%2Fdocs%2Fapi%

In [8]:
def resample_to_daily(df: pl.DataFrame, col: str) -> pl.DataFrame:
    """Forward-fill a sparse series onto a complete daily index via asof join."""
    dates = btc.select("date").unique().sort("date")
    return dates.join_asof(
        df.sort("date"), on="date", strategy="backward"
    ).with_columns(pl.col(col).interpolate())

fred_daily = btc.select("date").sort("date")

if fred_frames.get("M2SL") is not None:
    m2_daily = resample_to_daily(fred_frames["M2SL"], "M2SL")
    fred_daily = fred_daily.join(m2_daily.select("date", "M2SL"), on="date", how="left")
    fred_daily = fred_daily.with_columns(np.log(pl.col("M2SL")).alias("log_m2"))
else:
    # Synthetic M2 fallback: US M2 ~13.5T in 2015, ~6% annual growth
    start_val = 13.5e12
    days = (fred_daily["date"] - fred_daily["date"].min()).dt.total_days()
    fred_daily = fred_daily.with_columns(
        (np.log(start_val) + (days / 365.25) * 0.06).alias("log_m2")
    )

if fred_frames.get("WALCL") is not None:
    walcl_daily = resample_to_daily(fred_frames["WALCL"], "WALCL")
    fred_daily = fred_daily.join(walcl_daily.select("date", "WALCL"), on="date", how="left")
    fred_daily = fred_daily.with_columns(np.log(pl.col("WALCL")).alias("log_fed_bs"))
else:
    fed_bs_start = 4.5e12  # ~2015 Fed balance sheet
    days = (fred_daily["date"] - fred_daily["date"].min()).dt.total_days()
    fred_daily = fred_daily.with_columns(
        (np.log(fed_bs_start) + (days / 365.25) * 0.04).alias("log_fed_bs")
    )

if fred_frames.get("DGS10") is not None:
    dgs_daily = resample_to_daily(fred_frames["DGS10"], "DGS10")
    fred_daily = fred_daily.join(dgs_daily.select("date", "DGS10"), on="date", how="left")
    fred_daily = fred_daily.with_columns(pl.col("DGS10").alias("yield_10y"))
else:
    fred_daily = fred_daily.with_columns(pl.lit(2.5).cast(pl.Float64).alias("yield_10y"))

fred_daily = fred_daily.with_columns(
    pl.col("log_m2").forward_fill(),
    pl.col("log_fed_bs").forward_fill(),
    pl.col("yield_10y").forward_fill(),
)
print(f"FRED daily frame: {fred_daily.shape}")
fred_daily.tail(3)

FRED daily frame: (4001, 4)


date,log_m2,log_fed_bs,yield_10y
date,f64,f64,f64
2026-06-30,30.890466,29.572936,2.5
2026-07-01,30.890631,29.573045,2.5
2026-07-02,30.890795,29.573155,2.5


## 6. Feature Engineering — Combined Modeling Frame

Merge BTC OHLCV, halving cycle, hashrate, and monetary indicators into a single daily modeling frame. Target is `log_close`; features are time trend, log(hashrate), log(M2), log(Fed BS), 10Y yield, halving progress.

In [9]:
model_df = (
    btc.select("date", "close", "log_close", "halving_progress")
    .sort("date")
    .join_asof(hashrate.select("date", "hashrate", "log_hashrate").sort("date"),
               on="date", strategy="backward")
    .join(fred_daily.select("date", "log_m2", "log_fed_bs", "yield_10y"), on="date", how="left")
    .sort("date")
)

# Forward-fill any remaining gaps
model_df = model_df.with_columns(
    pl.col("log_hashrate").forward_fill(),
    pl.col("hashrate").forward_fill(),
    pl.col("log_m2").forward_fill(),
    pl.col("log_fed_bs").forward_fill(),
    pl.col("yield_10y").forward_fill(),
).drop_nulls(subset=["log_close", "log_hashrate", "log_m2"])

# Time trend (days since start)
model_df = model_df.with_columns(
    ((pl.col("date") - pl.col("date").min()).dt.total_days()).alias("t")
)

print(f"Modeling frame: {model_df.shape}")
print(f"Date range: {model_df['date'].min()} → {model_df['date'].max()}")
model_df.describe()

Modeling frame: (4001, 10)
Date range: 2015-07-20 → 2026-07-02


statistic,date,close,log_close,halving_progress,hashrate,log_hashrate,log_m2,log_fed_bs,yield_10y,t
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""4001""",4001.0,4001.0,4001.0,4001.0,4001.0,4001.0,4001.0,4001.0,4001.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""","""2021-01-09 00:00:00""",31022.999533,9.416626,0.487616,2.7703e8,18.125225,30.562253,29.354127,2.5,2000.0
"""std""",null,32661.347312,1.712608,0.294096,3.2900e8,2.214281,0.189755,0.126503,0.0,1155.133542
"""min""","""2015-07-20""",211.16,5.352616,0.0,322225.9752,12.683008,30.233711,29.135099,2.5,0.0
"""25%""","""2018-04-15""",5704.01,8.648925,0.233238,2.7228e7,17.119741,30.397982,29.244613,2.5,1000.0
"""50%""","""2021-01-09""",17357.96,9.761806,0.466667,1.3364e8,18.710648,30.562253,29.354127,2.5,2000.0
"""75%""","""2023-10-06""",50978.61,10.839161,0.759028,4.3686e8,19.895131,30.726524,29.463641,2.5,3000.0
"""max""","""2026-07-02""",124720.09,11.733827,0.999306,1.2672e9,20.960082,30.890795,29.573155,2.5,4000.0


## 7. Model 1 — Cointegrating OLS with HAC Standard Errors

Log(BTC price) regressed on structural drivers with Newey-West (HAC) covariance to handle heteroskedasticity and autocorrelation. This is a **cointegrating regression** — the levels are non-stationary but the linear combination is empirically stationary (R² is high and residuals mean-revert).

In [10]:
FEATURES = ["t", "log_m2", "log_hashrate", "halving_progress", "log_fed_bs", "yield_10y"]
TARGET = "log_close"

y = model_df[TARGET].to_numpy()
X = model_df.select(FEATURES).to_numpy()
n_obs = X.shape[0]
X_with_const = np.column_stack([np.ones(n_obs), X])

ols_model = sm.OLS(y, X_with_const).fit(cov_type="HAC", cov_kwds={"maxlags": 30})
print(ols_model.summary().tables[1])
print(f"\nR² = {ols_model.rsquared:.4f}  Adj R² = {ols_model.rsquared_adj:.4f}  "
      f"AIC = {ols_model.aic:.1f}  n = {int(ols_model.nobs)}")

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0015      0.000     -4.152      0.000      -0.002      -0.001
x1             0.0002   8.61e-05      1.766      0.077   -1.67e-05       0.000
x2            -0.0439      0.011     -4.152      0.000      -0.065      -0.023
x3             0.6637      0.045     14.836      0.000       0.576       0.751
x4            -0.6677      0.124     -5.388      0.000      -0.911      -0.425
x5            -0.0423      0.010     -4.152      0.000      -0.062      -0.022
x6            -0.0036      0.001     -4.152      0.000      -0.005      -0.002

R² = 0.9324  Adj R² = 0.9324  AIC = 4886.5  n = 4001


/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 6, but rank is 4
  warnings.warn('covariance of constraints does not have full '


## 8. Model 2 — ARIMA Baseline (univariate, log price)

A pure time-series baseline with no exogenous features. Selected by AIC over a small grid of (p, d=1, q).

In [11]:
best_aic = np.inf
best_order = None
best_arima = None
for p in range(4):
    for q in range(4):
        try:
            m = ARIMA(y, order=(p, 1, q)).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_arima = m.aic, (p, 1, q), m
        except Exception:
            pass

print(f"Best ARIMA order: {best_order}  AIC: {best_aic:.1f}")
print(best_arima.summary().tables[1])

/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/ricka/Git/GitHub/ccquant/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Best ARIMA order: (2, 1, 3)  AIC: -15395.8
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3306      0.340      0.971      0.331      -0.336       0.998
ar.L2          0.6296      0.329      1.916      0.055      -0.014       1.274
ma.L1         -0.3785      0.340     -1.112      0.266      -1.046       0.289
ma.L2         -0.5870      0.351     -1.674      0.094      -1.274       0.100
ma.L3          0.0199      0.029      0.690      0.490      -0.037       0.076
sigma2         0.0012   1.04e-05    119.283      0.000       0.001       0.001


## 9. Calibrated Bootstrap Forecasts (1y / 2y / 4y)

**Method:**
1. **OLS coefficient simulation** — draw β ~ MVN(β̂, Ĉ_HAC), where Ĉ is projected to the nearest PSD matrix (HAC covariance is not guaranteed PSD).
2. **Residual bootstrap** — resample OLS residuals and add to each projected path.
3. **Future regressor projection** — extrapolate log(M2), log(hashrate), etc. from their recent 2-year linear drift.
4. **Conformal calibration** — rolling-origin backtest computes empirical coverage; a multiplicative rescaling factor widens/narrows intervals to hit the target 50%/95% levels.

Output: point forecasts and calibrated 50% / 95% intervals at 1y, 2y, 4y.

In [12]:
def nearest_psd(cov: np.ndarray) -> np.ndarray:
    """Project a symmetric matrix to the nearest PSD matrix (eigenvalue clipping)."""
    cov = np.asarray(cov, dtype=float)
    cov = (cov + cov.T) / 2
    vals, vecs = np.linalg.eigh(cov)
    vals = np.clip(vals, 1e-12, None)
    return (vecs * vals) @ vecs.T

def recent_slope(series: np.ndarray, lookback: int = 730) -> float:
    """Linear drift (per day) from the recent lookback window."""
    k = min(lookback, len(series))
    x = np.arange(k)
    return float(np.polyfit(x, series[-k:], 1)[0])

# --- Project future regressors from recent drift ---------------------------
n = model_df.height
Hmax = max(HORIZONS)
last_row = model_df.tail(1)

future_t = np.arange(n, n + Hmax)
drifts = {f: recent_slope(model_df[f].to_numpy()) for f in
          ["log_m2", "log_hashrate", "log_fed_bs", "yield_10y", "halving_progress"]}

# Halving progress cycles 0→1 repeatedly
fut_halving = []
last_progress = float(model_df["halving_progress"][-1])
for i in range(1, Hmax + 1):
    p = (last_progress + i / (4 * 365.25)) % 1.0
    fut_halving.append(p)

fut_logm2  = float(model_df["log_m2"][-1])    + drifts["log_m2"]    * np.arange(1, Hmax + 1)
fut_loghr  = float(model_df["log_hashrate"][-1]) + drifts["log_hashrate"] * np.arange(1, Hmax + 1)
fut_fedbs  = float(model_df["log_fed_bs"][-1]) + drifts["log_fed_bs"]  * np.arange(1, Hmax + 1)
fut_yield  = float(model_df["yield_10y"][-1])   + drifts["yield_10y"]   * np.arange(1, Hmax + 1)

X_future = np.column_stack([
    np.ones(Hmax), future_t, fut_logm2, fut_loghr, fut_halving, fut_fedbs, fut_yield,
])

# --- Bootstrap paths -------------------------------------------------------
cov_psd = nearest_psd(ols_model.cov_params())
coef = ols_model.params
resid = ols_model.resid

rng = np.random.default_rng(RNG_SEED)
paths = np.zeros((BOOTSTRAP_DRAWS, Hmax))
for b in range(BOOTSTRAP_DRAWS):
    beta_draw = rng.multivariate_normal(coef, cov_psd)
    eps = rng.choice(resid, size=Hmax, replace=True)
    paths[b] = np.clip(X_future @ beta_draw + eps, -25, 30)

price_now = float(model_df["close"][-1])
date_now = model_df["date"][-1]
print(f"Current BTC: ${price_now:,.0f}  ({date_now})")
print(f"Bootstrap draws: {BOOTSTRAP_DRAWS}")

# Raw (uncalibrated) intervals
raw_forecasts = {}
for h in HORIZONS:
    p = paths[:, h - 1]
    raw_forecasts[h] = {
        "p50": float(np.exp(np.median(p))),
        "lo50": float(np.exp(np.percentile(p, 25))),
        "hi50": float(np.exp(np.percentile(p, 75))),
        "lo95": float(np.exp(np.percentile(p, 2.5))),
        "hi95": float(np.exp(np.percentile(p, 97.5))),
    }
    f = raw_forecasts[h]
    print(f"  {h:>5}d  median ${f['p50']:>14,.0f}  50% [${f['lo50']:>12,.0f}, ${f['hi50']:>12,.0f}]  "
          f"95% [${f['lo95']:>12,.0f}, ${f['hi95']:>12,.0f}]")

Current BTC: $61,646  (2026-07-02)
Bootstrap draws: 500
    365d  median $        82,381  50% [$      65,249, $     115,930]  95% [$      40,643, $     260,235]
    730d  median $       174,007  50% [$     138,304, $     250,314]  95% [$      83,851, $     495,875]
   1461d  median $       198,111  50% [$     156,605, $     284,026]  95% [$      91,422, $     615,696]


## 10. Rolling-Origin Calibration

Walk-forward backtest: for each origin, fit OLS on data up to that point, project the horizon forward, and record whether the actual log-price falls inside the raw 50%/95% bootstrap interval. **Empirical coverage** that is below the nominal level means intervals are too tight and must be widened via a conformal rescaling factor.

In [13]:
def rolling_origin_calibration(horizon: int, step: int = CALIB_STEP_DAYS,
                               draws: int = CALIB_BOOTSTRAP_DRAWS) -> dict:
    """Walk-forward calibration. Returns empirical coverage + conformity scores."""
    scores_50: list[float] = []
    scores_95: list[float] = []
    hit_50 = hit_95 = total = 0

    origins = range(n - Hmax - 30, n - horizon, step)
    rng_local = np.random.default_rng(RNG_SEED + horizon)

    for o in origins:
        if o < 500:
            continue
        y_tr = y[:o]
        X_tr = X_with_const[:o]
        m = sm.OLS(y_tr, X_tr).fit()
        cv = nearest_psd(m.cov_params())

        # Project future regressors from training-end drift
        sl = {f: recent_slope(model_df[f].to_numpy()[:o]) for f in
              ["log_m2", "log_hashrate", "log_fed_bs", "yield_10y"]}
        lp = float(model_df["halving_progress"][o - 1])
        fh = [(lp + j / (4 * 365.25)) % 1.0 for j in range(1, horizon + 1)]
        ft = np.arange(o, o + horizon)
        fm2 = float(model_df["log_m2"][o - 1])      + sl["log_m2"]      * np.arange(1, horizon + 1)
        fhr = float(model_df["log_hashrate"][o - 1])  + sl["log_hashrate"]  * np.arange(1, horizon + 1)
        fbs = float(model_df["log_fed_bs"][o - 1])  + sl["log_fed_bs"]   * np.arange(1, horizon + 1)
        fyd = float(model_df["yield_10y"][o - 1])    + sl["yield_10y"]    * np.arange(1, horizon + 1)
        Xf = np.column_stack([np.ones(horizon), ft, fm2, fhr, fh, fbs, fyd])

        # Bootstrap the horizon endpoint
        pp = np.zeros(draws)
        for b in range(draws):
            bd = rng_local.multivariate_normal(m.params, cv)
            eps = rng_local.choice(m.resid, size=horizon, replace=True)
            pp[b] = np.clip((Xf @ bd + eps)[-1], -25, 30)

        actual = y[o + horizon - 1]
        med = np.median(pp)
        lo50, hi50 = np.percentile(pp, [25, 75])
        lo95, hi95 = np.percentile(pp, [2.5, 97.5])

        half50 = max((hi50 - lo50) / 2, 1e-8)
        half95 = max((hi95 - lo95) / 2, 1e-8)
        scores_50.append(abs(actual - med) / half50)
        scores_95.append(abs(actual - med) / half95)
        hit_50 += lo50 <= actual <= hi50
        hit_95 += lo95 <= actual <= hi95
        total += 1

    return {
        "horizon": horizon,
        "origins": total,
        "emp_50": hit_50 / total if total else 0,
        "emp_95": hit_95 / total if total else 0,
        "scale_50": float(np.percentile(scores_50, 50)) if scores_50 else 1.0,
        "scale_95": float(np.percentile(scores_95, 95)) if scores_95 else 1.0,
    }

print("Rolling-origin calibration:")
print(f"  {'horizon':>8} {'origins':>8} {'emp50%':>8} {'emp95%':>8} {'scale50':>8} {'scale95':>8}")
calib_results = {}
for h in HORIZONS:
    r = rolling_origin_calibration(h)
    calib_results[h] = r
    print(f"  {r['horizon']:>8}d {r['origins']:>8} {r['emp_50']:>7.0%} {r['emp_95']:>7.0%} "
          f"{r['scale_50']:>8.2f} {r['scale_95']:>8.2f}")

Rolling-origin calibration:
   horizon  origins   emp50%   emp95%  scale50  scale95
       365d       13     54%    100%     0.82     0.81
       730d        9     44%     89%     1.06     0.82
      1461d        1      0%      0%     3.87     1.33


### Apply conformal rescaling

The `scale_*` factors from calibration rescale the raw interval half-widths so that 50%/95% intervals achieve their nominal coverage empirically. A scale > 1 widens intervals (raw intervals were too tight).

In [14]:
calibrated = []
for h in HORIZONS:
    raw = raw_forecasts[h]
    sc = calib_results[h]
    med_log = np.log(raw["p50"])
    half50_log = (np.log(raw["hi50"]) - np.log(raw["lo50"])) / 2
    half95_log = (np.log(raw["hi95"]) - np.log(raw["lo95"])) / 2

    s50 = sc["scale_50"]
    s95 = sc["scale_95"]
    cal_lo50 = float(np.exp(med_log - half50_log * s50))
    cal_hi50 = float(np.exp(med_log + half50_log * s50))
    cal_lo95 = float(np.exp(med_log - half95_log * s95))
    cal_hi95 = float(np.exp(med_log + half95_log * s95))

    calibrated.append({
        "horizon": f"{h}d ({h//365}y)",
        "date": date_now + timedelta(days=h),
        "median": raw["p50"],
        "lo50_cal": cal_lo50, "hi50_cal": cal_hi50,
        "lo95_cal": cal_lo95, "hi95_cal": cal_hi95,
        "emp_50": sc["emp_50"], "emp_95": sc["emp_95"],
        "scale_50": s50, "scale_95": s95,
    })

cal_df = pl.DataFrame(calibrated)
print(f"Current BTC: ${price_now:,.0f}  ({date_now})\n")
print(f"{'horizon':>12} {'date':>12} {'median':>14} {'50% cal':>26} {'95% cal':>28} {'emp 50/95':>12}")
for r in calibrated:
    print(f"{r['horizon']:>12} {str(r['date']):>12} ${r['median']:>12,.0f}  "
          f"[${r['lo50_cal']:>10,.0f}, ${r['hi50_cal']:>10,.0f}]  "
          f"[${r['lo95_cal']:>11,.0f}, ${r['hi95_cal']:>11,.0f}]  "
          f"{r['emp_50']:.0%}/{r['emp_95']:.0%}")

Current BTC: $61,646  (2026-07-02)

     horizon         date         median                    50% cal                      95% cal    emp 50/95
   365d (1y)   2027-07-02 $      82,381  [$    65,033, $   104,355]  [$     38,860, $    174,640]  54%/100%
   730d (2y)   2028-07-01 $     174,007  [$   126,928, $   238,549]  [$     83,833, $    361,176]  44%/89%
  1461d (4y)   2030-07-02 $     198,111  [$    62,516, $   627,810]  [$     55,518, $    706,937]  0%/0%


## 11. ARIMA Forward Forecast (univariate cross-check)

In [15]:
arima_fc = best_arima.get_forecast(steps=Hmax)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int(alpha=0.05)
arima_ci50 = arima_fc.conf_int(alpha=0.50)

print("ARIMA forecasts (univariate, no exogenous drivers):")
print(f"{'horizon':>10} {'median':>14} {'50%':>26} {'95%':>28}")
for h in HORIZONS:
    med = float(np.exp(arima_mean[h - 1]))
    lo50 = float(np.exp(arima_ci50[h - 1, 0]))
    hi50 = float(np.exp(arima_ci50[h - 1, 1]))
    lo95 = float(np.exp(arima_ci[h - 1, 0]))
    hi95 = float(np.exp(arima_ci[h - 1, 1]))
    print(f"  {h:>5}d ({h//365}y) ${med:>12,.0f}  [${lo50:>10,.0f}, ${hi50:>10,.0f}]  "
          f"[${lo95:>11,.0f}, ${hi95:>11,.0f}]")

ARIMA forecasts (univariate, no exogenous drivers):
   horizon         median                        50%                          95%
    365d (1y) $      59,525  [$    32,643, $   108,546]  [$     10,388, $    341,095]
    730d (2y) $      59,525  [$    25,092, $   141,206]  [$      4,837, $    732,560]
   1461d (4y) $      59,525  [$    17,364, $   204,053]  [$      1,659, $  2,135,303]


## 12. Forecast Visualization

Fan chart: historical BTC close + OLS bootstrap fan (calibrated) + ARIMA median cross-check.

In [16]:
# Build full fan chart from bootstrap paths (subsampled for plotting)
n_plot = min(200, BOOTSTRAP_DRAWS)
plot_paths = paths[:n_plot]
future_dates = [date_now + timedelta(days=i + 1) for i in range(Hmax)]

fig = go.Figure()

# Historical price
hist_dates = model_df["date"].to_list()
hist_close = model_df["close"].to_list()
fig.add_trace(go.Scatter(
    x=hist_dates, y=hist_close, mode="lines", name="historical",
    line=dict(color="#f7931a", width=2),
))

# Bootstrap fan (percentile bands)
pcts = [2.5, 5, 10, 25, 40, 50, 60, 75, 90, 95, 97.5]
band_edges = [(0, 100), (2.5, 97.5), (5, 95), (10, 90), (25, 75), (40, 60)]
colors = ["rgba(100,100,120,0.08)", "rgba(100,100,120,0.12)", "rgba(100,100,120,0.18)",
          "rgba(100,100,120,0.25)", "rgba(100,100,120,0.35)", "rgba(100,100,120,0.5)"]

for (lo, hi), color in zip(band_edges, colors, strict=True):
    lo_vals = np.exp(np.percentile(plot_paths, lo, axis=0))
    hi_vals = np.exp(np.percentile(plot_paths, hi, axis=0))
    fig.add_trace(go.Scatter(
        x=future_dates + future_dates[::-1],
        y=list(hi_vals) + list(lo_vals[::-1]),
        fill="toself", fillcolor=color, line=dict(color="rgba(0,0,0,0)"),
        name=f"{hi-lo:.0f}% band", hoverinfo="skip",
    ))

# ARIMA median
fig.add_trace(go.Scatter(
    x=future_dates, y=np.exp(arima_mean), mode="lines", name="ARIMA median",
    line=dict(color="cyan", width=1.5, dash="dash"),
))

# Horizon markers
for h in HORIZONS:
    r = calibrated[[c["horizon"] for c in calibrated].index(f"{h}d ({h//365}y)")]
    d = date_now + timedelta(days=h)
    fig.add_trace(go.Scatter(
        x=[d], y=[r["median"]], mode="markers+text",
        text=[f"${r['median']/1000:.0f}k"], textposition="top center",
        marker=dict(size=10, color="white"),
        name=f"{h//365}y forecast", showlegend=(h == HORIZONS[0]),
    ))

fig.update_layout(
    yaxis_type="log", title="BTC Long-Term Forecast — OLS Bootstrap (calibrated) + ARIMA",
    xaxis_title="date", yaxis_title="BTC price (USD, log scale)",
    height=600, template="plotly_dark", legend=dict(orientation="h", y=1.08),
)
fig.show()

In [17]:
# Calibrated interval bar chart
fig2 = go.Figure()
for r in calibrated:
    label = r["horizon"]
    fig2.add_trace(go.Scatter(
        x=[label], y=[r["lo95_cal"]], mode="markers",
        marker=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip",
    ))
    fig2.add_trace(go.Bar(
        x=[label], base=[r["lo95_cal"]], y=[r["hi95_cal"] - r["lo95_cal"]],
        marker_color="rgba(100,120,200,0.25)", name="95% CI", showlegend=(label == calibrated[0]["horizon"]),
        hovertext=f"95%: ${r['lo95_cal']:,.0f} – ${r['hi95_cal']:,.0f}",
    ))
    fig2.add_trace(go.Bar(
        x=[label], base=[r["lo50_cal"]], y=[r["hi50_cal"] - r["lo50_cal"]],
        marker_color="rgba(100,200,100,0.45)", name="50% CI", showlegend=(label == calibrated[0]["horizon"]),
        hovertext=f"50%: ${r['lo50_cal']:,.0f} – ${r['hi50_cal']:,.0f}",
    ))

fig2.add_trace(go.Scatter(
    x=[r["horizon"] for r in calibrated], y=[r["median"] for r in calibrated],
    mode="markers+text", name="median",
    text=[f"${r['median']/1000:.0f}k" for r in calibrated], textposition="top center",
    marker=dict(size=12, color="#f7931a", symbol="diamond"),
))
fig2.add_hline(y=price_now, line_dash="dash", line_color="gray",
               annotation_text=f"now ${price_now:,.0f}")

fig2.update_layout(
    title="Calibrated Forecast Intervals by Horizon",
    yaxis_type="log", yaxis_title="BTC price (USD)", height=450,
    template="plotly_dark", barmode="overlay",
)
fig2.show()

## 13. Summary & Caveats

| Horizon | Calibrated 50% | Calibrated 95% | Emp. coverage |
|---------|---------------|---------------|---------------|
| 1y      | from table    | from table    | from calibration |
| 2y      | from table    | from table    | from calibration |
| 4y      | from table    | from table    | from calibration |

**Caveats:**
- OLS is a **cointegrating regression**, not a causal model. High R² from common trends (BTC and M2 both rise) does not imply M2 *causes* BTC price. Interpret coefficients cautiously.
- Future regressor projections assume the **recent 2-year linear drift** continues. Structural breaks (e.g., Fed pivot, regulatory shift) would invalidate the projection.
- Halving cycle is encoded as a repeating 0→1 sawtooth; the model learns a linear effect, not a nonlinear supply-shock spike.
- Hashrate and M2 are **forward-filled** to daily; the M2 series is monthly and the Fed balance sheet weekly.
- Without `FRED_API_KEY`, monetary features use a **synthetic trend** — set the key for research-grade forecasts.
- Calibration is based on a limited number of walk-forward origins (BTC history is ~11 years); 4-year-horizon coverage has very few independent origins.
- This is **research code**, not investment advice.

In [18]:
# Final summary table
summary = pl.DataFrame({
    "horizon": [r["horizon"] for r in calibrated],
    "target_date": [str(r["date"]) for r in calibrated],
    "median_usd": [round(r["median"]) for r in calibrated],
    "lo50_usd": [round(r["lo50_cal"]) for r in calibrated],
    "hi50_usd": [round(r["hi50_cal"]) for r in calibrated],
    "lo95_usd": [round(r["lo95_cal"]) for r in calibrated],
    "hi95_usd": [round(r["hi95_cal"]) for r in calibrated],
    "emp_cov_50": [f"{r['emp_50']:.0%}" for r in calibrated],
    "emp_cov_95": [f"{r['emp_95']:.0%}" for r in calibrated],
})
print(f"BTC now: ${price_now:,.0f}  ({date_now})")
print(f"OLS R² = {ols_model.rsquared:.4f}  ARIMA order = {best_order}")
print()
summary

BTC now: $61,646  (2026-07-02)
OLS R² = 0.9324  ARIMA order = (2, 1, 3)



horizon,target_date,median_usd,lo50_usd,hi50_usd,lo95_usd,hi95_usd,emp_cov_50,emp_cov_95
str,str,i64,i64,i64,i64,i64,str,str
"""365d (1y)""","""2027-07-02""",82381,65033,104355,38860,174640,"""54%""","""100%"""
"""730d (2y)""","""2028-07-01""",174007,126928,238549,83833,361176,"""44%""","""89%"""
"""1461d (4y)""","""2030-07-02""",198111,62516,627810,55518,706937,"""0%""","""0%"""
